In [1]:
import gensim
from gensim.models import Word2Vec, FastText
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. Текстовый корпус банковских обращений
# ==========================================
# Токенизированные предложения (списки слов)
corpus = [
    ["заблокировали", "дебетовую", "карту", "без", "предупреждения"],
    ["не", "могу", "перевести", "деньги", "через", "приложение"],
    ["приложение", "вылетает", "при", "входе", "в", "личный", "кабинет"],
    ["подозрительное", "списание", "средств", "со", "счета", "мошенники"],
    ["карта", "заблокирована", "банком", "как", "разблокировать"],
    ["быстрый", "и", "удобный", "перевод", "денег", "на", "кредитную", "карту"],
    ["ужасный", "сервис", "поддержка", "не", "отвечает", "целый", "день"],
    ["украли", "деньги", "с", "кредитной", "карты", "списали", "все", "средства"],
    ["не", "работает", "перевод", "по", "номеру", "телефона"],
    ["требую", "вернуть", "деньги", "заблокировали", "счет", "мошенничество"]
] * 30  # Умножаем корпус для стабильности обучения

In [2]:
# ==========================================
# ЗАДАНИЕ 1: Обучение Word2Vec и FastText в Gensim
# ==========================================
# TODO 1:
# 1. Обучите модель Word2Vec (vector_size=32, window=3, min_count=1, sg=1 (Skip-gram), epochs=20)
# 2. Обучите модель FastText с точно такими же параметрами

w2v_model = Word2Vec(sentences=corpus, vector_size=32, window=3, min_count=1, sg=1, epochs=20)
ft_model = FastText(sentences=corpus, vector_size=32, window=3, min_count=1, sg=1, epochs=20)

In [3]:
# ==========================================
# ЗАДАНИЕ 2: Проверка OOV (Out-Of-Vocabulary) и синонимов
# ==========================================
# TODO 2:
# 1. Найдите топ-3 похожих слова к слову "карта" в Word2Vec через .wv.most_similar()
print(w2v_model.wv.most_similar('карта', topn=3), '\n')

# 2. Попробуйте проверить слово с опечаткой: "каарту" или "заблоокировали"
#    - Что произойдет при вызове w2v_model.wv["каарту"]? (подсказка: обработайте KeyError через try/except)
try:
  vec = w2v_model.wv["каарту"]
  print(vec)
except KeyError:
  print("Слова нет в словаре w2v\n")

#    - Что произойдет при вызове ft_model.wv["каарту"]?
ft_vec = ft_model.wv["каарту"]
print(ft_vec, '\n')

# 3. Выведите косинусное сходство между "дебетовую" и "кредитную" в обеих моделях (.wv.similarity)
print("Word2Vec", w2v_model.wv.similarity(w1="дебетовую", w2="кредитную"))

[('заблокирована', 0.9895570874214172), ('как', 0.9881019592285156), ('банком', 0.9870731830596924)] 

Слова нет в словаре w2v

[ 0.06742418 -0.05377142  0.1466709   0.07277422  0.02089785 -0.00922641
 -0.10672273 -0.13078627  0.04839511 -0.02890389  0.02841568 -0.03971962
 -0.02184662 -0.03543369 -0.00823298  0.01664742 -0.02593401  0.05671477
  0.10411137  0.05266158 -0.13082491  0.14679573  0.10859856 -0.08426955
  0.04044362  0.25709546 -0.21945617  0.0439407  -0.07694025 -0.1404022
  0.02279696  0.11589578] 

Word2Vec 0.9634765


In [4]:
# ==========================================
# ЗАДАНИЕ 3: Загрузка матрицы весов FastText в PyTorch nn.Embedding
# ==========================================
# TODO 3:
# 1. Создайте словарь id2word (список всех слов из ft_model.wv.index_to_key)
id2word = ft_model.wv.index_to_key
print(id2word[0:5])

# 2. Добавьте в начало словаря спец-токены: "<pad>" (индекс 0) и "<unk>" (индекс 1)
id2word = ['<pad>', '<unk>'] + list(id2word)
word2id = {w: i for i, w in enumerate(id2word)}

# 3. Сформируйте numpy-матрицу весов weights_matrix формы [vocab_size, vector_size=32]:
#    - Для <pad> заполните строку нулями
#    - Для <unk> заполните случайными числами или средним вектором
#    - Для остальных слов скопируйте их векторы из ft_model.wv[word]
vocab_size = len(id2word)
vector_size = ft_model.wv.vector_size

weights_matrix = np.zeros((vocab_size, vector_size), dtype=np.float32)
weights_matrix[1] = ft_model.wv.vectors.mean(axis=0)

for i, word in enumerate(id2word[2:], start=2):
  weights_matrix[i] = ft_model.wv[word]
print(weights_matrix.shape)

# 4. Инициализируйте PyTorch слой nn.Embedding.from_pretrained(torch.FloatTensor(weights_matrix), freeze=True)
embedding = nn.Embedding.from_pretrained(
    torch.FloatTensor(weights_matrix),
    freeze=True,
    padding_idx=0
)
print(embedding.weight.shape)
print(embedding.weight.requires_grad) # потому что заморозили градиенты для обучения

# 5. Передайте тензор токенов tensor([[2, 3, 0]]) через ваш embedding слой и выведите полученную форму (shape).
tokens = torch.tensor([[2, 3, 0]])
print("tokens.shape", tokens.shape)

embed = embedding(tokens)
print(embed.shape)


['деньги', 'не', 'перевод', 'приложение', 'карту']
(58, 32)
torch.Size([58, 32])
False
tokens.shape torch.Size([1, 3])
torch.Size([1, 3, 32])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ==========================================
# 0. Подготовка датасета (3 класса)
# 0: Безопасность / Блокировки
# 1: Переводы и платежи
# 2: Технические проблемы и сервис
# ==========================================
dataset = [
    ("заблокировали дебетовую карту без предупреждения", 0),
    ("подозрительное списание средств со счета мошенники", 0),
    ("украли деньги с кредитной карты списали все средства", 0),
    ("не могу перевести деньги через приложение", 1),
    ("быстрый и удобный перевод денег на кредитную карту", 1),
    ("не работает перевод по номеру телефона", 1),
    ("приложение вылетает при входе в личный кабинет", 2),
    ("ужасный сервис поддержка не отвечает целый день", 2),
]

# ==========================================
# ЗАДАНИЕ 1: Функция подготовки входных тензоров (Padding & Indexing)
# ==========================================
# TODO 1: Напиши функцию, которая превращает предложение в список ID фиксированной длины:
# - Токенизирует строку по пробелам
# - Переводит слова в ID с помощью word2id (если слова нет в словаре -> берем ID для "<unk>")
# - Дополняет последовательность нулями (ID для "<pad>") до max_len или обрезает, если длина > max_len

def text_to_ids(text, word2id, max_len=8):
    tokens = text.split()
    ids = [word2id.get(w, word2id['<unk>']) for w in tokens]
    if len(ids)>max_len:
        ids = ids[:max_len]
    elif len(ids)<max_len:
        ids += [0]*(max_len-len(ids))

    return ids

# Проверка: преобразуй весь датасет в тензоры X (индексы) и y (метки)

X_data = [text_to_ids(text, word2id, max_len=8) for text, label in dataset]
X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor([label for _, label in dataset], dtype=torch.long)

# ==========================================
# ЗАДАНИЕ 2: Архитектурный класс PyTorch модели
# ==========================================
# TODO 2: Реализуй классификатор с усреднением эмбеддингов (Global Average Pooling):

class TextClassifier(nn.Module):
    def __init__(self, weights_matrix, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(weights_matrix), freeze=True, padding_idx=0)

        self.fc1 = nn.Linear(32, 16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        self.fc2 = nn.Linear(16, num_classes)

        self._init_weights()

    def forward(self, x):
        emb = self.embedding(x)
        pooled = emb.mean(dim=1)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        logits = self.fc2(out)

        return logits

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
    



# ==========================================
# ЗАДАНИЕ 3: Обучение модели (Training Loop)
# ==========================================
# TODO 3:
# 1. Инициализируй модель model = TextClassifier(weights_matrix)
# 2. Определи функцию потерь criterion = nn.CrossEntropyLoss()
# 3. Определи оптимизатор optimizer = optim.Adam(model.parameters(), lr=0.01)
# 4. Напиши цикл обучения на 50 эпох

model = TextClassifier(weights_matrix=weights_matrix)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01)

epochs = 50

def train_epoch(model, criterion, optimizer, device, data):
    model.train()
    total_loss = 0.0

    for string, label in data:
        ids = torch.tensor([text_to_ids(string, word2id)])
        ten_label = torch.tensor([label])
        ids = ids.to(device)
        ten_label = ten_label.to(device)

        optimizer.zero_grad()

        logits = model(ids)
        loss = criterion(logits, ten_label)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
    return total_loss/len(data)

for i in range(epochs):
    loss = train_epoch(model, criterion, optimizer, device, dataset)
    if (i+1)%10 == 0:
        print(i+1, "эпоха, ошибка =", loss)

10 эпоха, ошибка = 1.0282458141446114
20 эпоха, ошибка = 1.1825650110840797
30 эпоха, ошибка = 1.3611237183213234
40 эпоха, ошибка = 1.1066897511482239
50 эпоха, ошибка = 0.9936694875359535
